1_statistical_engine.ipynb.ipynb

📁 Cell 1: Setup & Imports

In [14]:
# # ============================================================
# # GOAT-Net — Phase 2: Statistical Engine (Ultimate Version)
# # ============================================================
# # ============================================================
# # GOAT-Net — Standard Session Setup
# # ============================================================
# import os
# import sys
# import random
# import numpy as np
# from pathlib import Path
# from google.colab import drive

# # 0. Set seeds for reproducibility
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)

# # 1. Mount Drive
# drive.mount("/content/drive", force_remount=False)

# # 2. Clone / Update repository into /content
# PROJECT = "GOAT-Net"
# REPO = Path("/content") / PROJECT

# if not REPO.exists():
#     print("🔄 Cloning repository...")
#     !git clone https://github.com/AtiX-Algo/{PROJECT}.git {REPO}
# else:
#     print("🔄 Updating repository...")
#     %cd {REPO}
#     !git pull origin main || true

# %cd {REPO}
# sys.path.append(str(REPO / "src"))

# # 3. Install requirements
# if (REPO / "requirements.txt").exists():
#     !pip install -q -r requirements.txt

# # 4. Define data paths (Drive)
# DATA_ROOT = Path("/content/drive/MyDrive") / PROJECT

# # For backward compatibility (so old code using ROOT still works)
# ROOT = DATA_ROOT

# PATHS = {
#     "repo": REPO,
#     "raw": DATA_ROOT / "data" / "raw",
#     "processed": DATA_ROOT / "data" / "processed",
#     "metadata": DATA_ROOT / "metadata",
#     "models": DATA_ROOT / "models",
#     "output": DATA_ROOT / "data" / "processed" / "statistical",
# }

# # Create directories
# for p in PATHS.values():
#     if not str(p).endswith(('.parquet', '.csv')):
#         p.mkdir(parents=True, exist_ok=True)

# # Set environment variable for DatasetManager
# os.environ["GOAT_DATA_ROOT"] = str(DATA_ROOT)

# print("\n✅ Session ready")
# print(f"   Code repo    : {PATHS['repo']}")
# print(f"   Data root    : {DATA_ROOT}")
# print(f"   ROOT (alias) : {ROOT}")

In [15]:
# ============================================================
# GOAT-Net — Phase 2: Statistical Engine (Fixed Setup)
# ============================================================
import os, sys, random
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 1. Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# 3. Set Working Directory (NO GIT PULLING OR CLONING)
PROJECT = "GOAT-Net"
REPO = Path("/content/drive/MyDrive") / PROJECT
os.chdir(REPO)
print(f"📁 Working directory securely set to: {REPO}")

# 4. Fix Python Path
SRC_DIR = REPO / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 5. Clear cache to prevent ModuleNotFoundError
for m in list(sys.modules):
    if m == "data" or m.startswith("data.") or m.startswith("goatnet."):
        del sys.modules[m]

# 6. Install Dependencies
print("⏳ Installing statistical dependencies...")
!pip install -q pyarrow tqdm networkx mplsoccer

# 7. Setup Paths
DATA_ROOT = REPO
PATHS = {
    "repo": REPO,
    "raw": DATA_ROOT / "data" / "raw",
    "processed": DATA_ROOT / "data" / "processed",
    "metadata": DATA_ROOT / "metadata",
    "models": DATA_ROOT / "models",
    "output": DATA_ROOT / "data" / "processed" / "statistical"
}

# Create directories
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete. Ready for Cell 2!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Working directory securely set to: /content/drive/MyDrive/GOAT-Net
⏳ Installing statistical dependencies...
✅ Setup complete. Ready for Cell 2!


📊 Cell 2: Load All Raw Datasets

In [16]:
# # ============================================================
# # GOAT-Net — Phase 2: Statistical Engine
# # ============================================================
# import warnings
# warnings.filterwarnings("ignore")

# import pandas as pd

# # DatasetManager now uses the PATHS from the setup cell
# import data.dataset_manager as dm

# # Output directory (already defined in PATHS)
# OUTPUT = PATHS["output"]

# print("📊 Loading datasets...")
# events = dm.load_statsbomb_events()
# fbref = dm.load_fbref_stats()
# understat = dm.load_understat_xg()
# tm = dm.load_transfermarkt()

# # Safely load FIFA – now using PATHS["raw"] instead of ROOT
# if hasattr(dm, 'load_fifa_ratings'):
#     fifa = dm.load_fifa_ratings()
# else:
#     fifa_path = PATHS["raw"] / "optional" / "fifa" / "tier1_fifa_ratings.parquet"
#     if fifa_path.exists():
#         fifa = pd.read_parquet(fifa_path)
#         mapping = dm.load_player_mapping()
#         fifa = fifa.merge(mapping[["fifa_name", "canonical_id"]],
#                           left_on="short_name", right_on="fifa_name", how="inner")
#         print(f"FIFA merge matched {len(fifa)} rows out of the panel — "     # ADD THIS LINE
#       f"if this looks low, short_name formatting probably doesn't match fifa_name exactly.")
#     else:
#         fifa = pd.DataFrame()

# before_merge = len(pd.read_parquet(fifa_path))

# print("="*60)
# print("FIFA Merge Validation")
# print("="*60)
# print(f"Rows before merge : {before_merge}")
# print(f"Rows after merge  : {len(fifa)}")
# print(f"Unique players    : {fifa['canonical_id'].nunique()}")

# print(f"StatsBomb events  : {len(events):>6} rows")
# print(f"FBref stats       : {len(fbref):>6} rows")
# print(f"Understat xG      : {len(understat):>6} rows")
# print(f"Transfermarkt     : {len(tm):>6} rows")
# print(f"FIFA ratings      : {len(fifa):>6} rows")

In [17]:
# ============================================================
# Cell 2: Load All Raw Datasets
# ============================================================
import warnings
warnings.filterwarnings("ignore")
import pandas as pd

# FIX: Import DatasetManager using the NEW 'goatnet' namespace!
import data.dataset_manager as dm

# Output directory
OUTPUT = PATHS["output"]

print("📊 Loading datasets...")
events = dm.load_statsbomb_events()
fbref = dm.load_fbref_stats()
understat = dm.load_understat_xg()
tm = dm.load_transfermarkt()

# Safely load FIFA
if hasattr(dm, 'load_fifa_ratings'):
    fifa = dm.load_fifa_ratings()
    before_merge = len(fifa)
else:
    fifa_path = PATHS["raw"] / "optional" / "fifa" / "tier1_fifa_ratings.parquet"
    if fifa_path.exists():
        before_merge = len(pd.read_parquet(fifa_path))
        fifa = pd.read_parquet(fifa_path)
        mapping = dm.load_player_mapping()
        fifa = fifa.merge(mapping[["fifa_name", "canonical_id"]],
                          left_on="short_name", right_on="fifa_name", how="inner")
        print(f"FIFA merge matched {len(fifa)} rows out of the panel — "
              f"if this looks low, short_name formatting probably doesn't match fifa_name exactly.")
    else:
        fifa = pd.DataFrame()
        before_merge = 0

print("="*60)
print("FIFA Merge Validation")
print("="*60)
print(f"Rows before merge : {before_merge}")
print(f"Rows after merge  : {len(fifa)}")
if not fifa.empty and 'canonical_id' in fifa.columns:
    print(f"Unique players    : {fifa['canonical_id'].nunique()}")

print(f"StatsBomb events  : {len(events):>6} rows")
print(f"FBref stats       : {len(fbref):>6} rows")
print(f"Understat xG      : {len(understat):>6} rows")
print(f"Transfermarkt     : {len(tm):>6} rows")
print(f"FIFA ratings      : {len(fifa):>6} rows")

📊 Loading datasets...
FIFA merge matched 58 rows out of the panel — if this looks low, short_name formatting probably doesn't match fifa_name exactly.
FIFA Merge Validation
Rows before merge : 58
Rows after merge  : 58
Unique players    : 6
StatsBomb events  :  49114 rows
FBref stats       :     90 rows
Understat xG      :     57 rows
Transfermarkt     :    359 rows
FIFA ratings      :     58 rows


🧹 Cell 3: Basic Cleaning & Filtering

In [18]:
print("\n" + "=" * 70)
print("🧹 CLEANING & FILTERING")
print("=" * 70)

def clean_fbref_columns(df):
    """Helper function to clean FBref multi-level headers. To be moved to DatasetManager later."""
    df_clean = df.copy()
    df_clean.columns = df_clean.columns.str.replace(r'^(Playing Time_|Performance_|Expected_)', '', regex=True)
    return df_clean

# --- 3.1 Drop duplicates (safely ignoring unhashable arrays in StatsBomb) ---
if not events.empty:
    if 'id' in events.columns:
        events = events.drop_duplicates(subset=['id']).reset_index(drop=True)
    else:
        events = events.loc[events.astype(str).drop_duplicates().index].reset_index(drop=True)

fbref = fbref.drop_duplicates().reset_index(drop=True) if not fbref.empty else fbref
understat = understat.drop_duplicates().reset_index(drop=True) if not understat.empty else understat
tm = tm.drop_duplicates().reset_index(drop=True) if not tm.empty else tm
fifa = fifa.drop_duplicates().reset_index(drop=True) if not fifa.empty else fifa

# --- 3.2 Filter FBref to only panel players (canonical_id not null) ---
if not fbref.empty:
    fbref = fbref.dropna(subset=['canonical_id', 'Season']).copy()

    # FIX: Use helper function to strip prefixes
    fbref = clean_fbref_columns(fbref)

    # --- 3.3 Fix Minutes (remove commas) ---
    if 'Min' in fbref.columns:
        fbref['Min'] = fbref['Min'].astype(str).str.replace(',', '')
        fbref['Min'] = pd.to_numeric(fbref['Min'], errors='coerce').fillna(0)

        # --- 3.4 Filter seasons with meaningful minutes (≥ 500) ---
        fbref_filtered = fbref[fbref['Min'] >= 500].copy()
        fbref_filtered['90s'] = fbref_filtered['Min'] / 90

        print(f"✅ FBref kept {len(fbref_filtered)} player-seasons with ≥ 500 minutes.")
        print(f"🗑️ Discarded {len(fbref) - len(fbref_filtered)} low‑minute seasons to prevent skewed per-90s.")
    else:
        print(f"⚠️ 'Min' column still not found. Columns are: {list(fbref.columns)}")
        fbref_filtered = fbref.copy()
else:
    fbref_filtered = pd.DataFrame()


🧹 CLEANING & FILTERING
✅ FBref kept 44 player-seasons with ≥ 500 minutes.
🗑️ Discarded 1 low‑minute seasons to prevent skewed per-90s.


🎯 Cell 4: Match‑Level Statistics (StatsBomb)

In [19]:
print("\n" + "=" * 70)
print("⚽ MATCH‑LEVEL STATISTICS")
print("=" * 70)

if events.empty:
    match_features = pd.DataFrame()
    print("⚠️ No StatsBomb events – skipping match-level stats.")
else:
    # Use shot_outcome_name for more accurate stats mapping
    shot_col = "shot_outcome_name" if "shot_outcome_name" in events.columns else "shot_outcome"

    # Group by (canonical_id, match_id)
    match_agg = events.groupby(["canonical_id", "common_name", "match_id"]).agg(
        shots=("type", lambda x: (x == "Shot").sum()),
        passes=("type", lambda x: (x == "Pass").sum()),
        goals=(shot_col, lambda x: (x == "Goal").sum() if shot_col in events.columns else 0),
        xG=("shot_statsbomb_xg", lambda x: x.sum() if "shot_statsbomb_xg" in events.columns else np.nan)
    ).reset_index()

    # Add competition and season
    if "competition_name" in events.columns and "season_name" in events.columns:
        comp_info = events.groupby("match_id")[["competition_name", "season_name"]].first().reset_index()
        match_features = match_agg.merge(comp_info, on="match_id", how="left")
    else:
        match_features = match_agg

    print(f"✅ Generated {len(match_features)} match appearances.")


⚽ MATCH‑LEVEL STATISTICS
✅ Generated 885 match appearances.


📈 Cell 5: Season‑Level Statistics Merge

In [20]:
print("\n" + "=" * 70)
print("📊 SEASON‑LEVEL STATISTICS")
print("=" * 70)

if fbref_filtered.empty:
    season_df = pd.DataFrame()
    print("⚠️ No FBref data – skipping season-level stats.")
else:
    # --- 5.1 Select relevant columns from FBref ---
    keep_cols = ["canonical_id", "common_name", "League", "Season", "Min",
                 "Gls", "Ast", "xG", "xA", "Sh", "SoT", "CrdY", "CrdR",
                 "Fls", "Fld", "Off", "Crs", "TklW", "PKwon", "PKcon", "OG", "Recov"]
    available = [c for c in keep_cols if c in fbref_filtered.columns]
    season_df = fbref_filtered[available].copy()
    season_df = season_df.rename(columns={"Min": "Minutes"})

    # --- 5.2 Add Understat data (Season-level Merge) ---
    if not understat.empty:
        us_cols = [c for c in ["xG", "xA", "xGChain", "xGBuildup"] if c in understat.columns]

        # Dynamically find the season column in understat
        season_col = next((c for c in ['Season', 'season', 'Year', 'year'] if c in understat.columns), None)

        if us_cols and season_col:

            # ==========================================================
            # Understat Source Validation
            # ==========================================================
            if "Source_File" in understat.columns:

                source_check = (
                    understat
                    .groupby("canonical_id")["Source_File"]
                    .nunique()
                )

                duplicated = source_check[source_check > 1]

                print("=" * 60)
                print("Understat Source Validation")
                print("=" * 60)

                print(f"Players with multiple source files: {len(duplicated)}")

                if len(duplicated):
                    display(
                        understat[
                            understat["canonical_id"].isin(duplicated.index)
                        ][["canonical_id", "Source_File"]]
                        .drop_duplicates()
                        .sort_values("canonical_id")
                    )

            # Existing code (leave unchanged)
            understat_agg = (
                understat
                .groupby(["canonical_id", season_col])[us_cols]
                .sum()
                .reset_index()
            )
            # Standardize naming to merge with FBRef
            understat_agg = understat_agg.rename(columns={season_col: "Season"})
            understat_agg = understat_agg.rename(columns={c: f"understat_{c}" for c in us_cols})

            # --- FIX: Harmonize Season types and formats ---
            # Extract the starting 4-digit year as a string to guarantee a match
            season_df['_season_match'] = season_df['Season'].astype(str).str.extract(r'^(\d{4})')[0]
            understat_agg['_season_match'] = understat_agg['Season'].astype(str).str.extract(r'^(\d{4})')[0]

            # Merge on canonical_id and the standardized season column
            season_df = season_df.merge(
                understat_agg.drop(columns=['Season']), # Drop Understat's raw Season column to prevent '_x' / '_y' duplication
                on=["canonical_id", "_season_match"],
                how="left"
            )

            # Clean up the temporary column
            season_df = season_df.drop(columns=['_season_match'])

        elif us_cols:
            print("⚠️ Understat missing 'Season' column. Cannot align xG accurately by season.")

    # --- 5.3 Add Transfermarkt valuations (safely sorted) ---
    if not tm.empty:
        tm_col = "market_value_in_eur" if "market_value_in_eur" in tm.columns else "market_value"
        if tm_col in tm.columns:
            tm_sorted = tm.sort_values("date") if "date" in tm.columns else tm
            tm_latest = tm_sorted.groupby("canonical_id").last().reset_index()
            season_df = season_df.merge(tm_latest[["canonical_id", tm_col]], on="canonical_id", how="left")

    # --- 5.4 Add FIFA ratings (explicitly sorted) ---
    if not fifa.empty and "overall" in fifa.columns:
        sort_col = next((col for col in ['year', 'fifa_version', 'fifa_update', 'date'] if col in fifa.columns), None)
        fifa_sorted = fifa.sort_values(sort_col) if sort_col else fifa
        fifa_latest = fifa_sorted.groupby('canonical_id').last().reset_index()
        season_df = season_df.merge(fifa_latest[["canonical_id", "overall", "potential"]], on="canonical_id", how="left")

    print(f"✅ Built season table with {len(season_df)} player‑seasons.")


📊 SEASON‑LEVEL STATISTICS
Understat Source Validation
Players with multiple source files: 0
✅ Built season table with 44 player‑seasons.


🧮 Cell 6: Derived Metrics (Efficiency & Clutch factors)

In [21]:
print("\n" + "=" * 70)
print("🧮 DERIVED METRICS")
print("=" * 70)

def add_derived_metrics(df):
    if df.empty:
        return df
    df = df.copy()

    calc_cols = ['Minutes', 'Gls', 'Ast', 'xG', 'xA', 'Sh', 'SoT']
    for col in calc_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(',', '')
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    # 1. Per90 metrics (Standardized names)
    if 'Gls' in df.columns:
        df['Goals_per90'] = np.where(df['Minutes'] > 0, (df['Gls'] / (df['Minutes'] / 90)).round(3), np.nan)
    if 'Ast' in df.columns:
        df['Assists_per90'] = np.where(df['Minutes'] > 0, (df['Ast'] / (df['Minutes'] / 90)).round(3), np.nan)

    for col in ['xG', 'xA', 'Sh', 'SoT']:
        if col in df.columns:
            df[f"{col}_per90"] = np.where(df['Minutes'] > 0, (df[col] / (df['Minutes'] / 90)).round(3), np.nan)

    # 2. Goal conversion (Shots → Goals)
    if 'Sh' in df.columns and 'Gls' in df.columns:
        df['Goal_Conv_Pct'] = np.where(df['Sh'] > 0, ((df['Gls'] / df['Sh']) * 100).round(2), np.nan)

    # 3. xG per shot (Shot Quality)
    if 'xG' in df.columns and 'Sh' in df.columns:
        df['xG_per_Shot'] = np.where(df['Sh'] > 0, (df['xG'] / df['Sh']).round(3), np.nan)

    # 4. Minutes per Goal
    if 'Gls' in df.columns:
        df['Mins_per_Goal'] = np.where(df['Gls'] > 0, (df['Minutes'] / df['Gls']).round(1), np.nan)

    # 5. Goals + Assists
    if 'Gls' in df.columns and 'Ast' in df.columns:
        df['Gls_Ast'] = df['Gls'].fillna(0) + df['Ast'].fillna(0)
        df['GoalContributions_per90'] = np.where(df['Minutes'] > 0, (df['Gls_Ast'] / (df['Minutes'] / 90)).round(3), np.nan)

    # 6. xG Overperformance (The "Clutch" Metric)
    if 'Gls' in df.columns and 'xG' in df.columns:
        df['xg_overperformance'] = (df['Gls'] - df['xG']).round(3)

    # 7. Shot Accuracy
    if 'SoT' in df.columns and 'Sh' in df.columns:
        df['ShotAccuracy'] = np.where(df['Sh'] > 0, (df['SoT'] / df['Sh']).round(3), np.nan)

    return df

season_df = add_derived_metrics(season_df)
print("✅ Derived metrics added to season table (Standardized Names).")


🧮 DERIVED METRICS
✅ Derived metrics added to season table (Standardized Names).


👤 Cell 7: Career Summary (Mathematically Corrected)

In [22]:
print("\n" + "=" * 70)
print("📋 CAREER SUMMARY (Summed Totals)")
print("=" * 70)

if season_df.empty:
    career_df = pd.DataFrame()
else:
    # Base Aggregation for Sums
    agg_dict = {
        'career_minutes': ('Minutes', 'sum'),
        'career_goals': ('Gls', 'sum'),
        'career_assists': ('Ast', 'sum'),
        'seasons_played': ('Season', 'count'),
    }

    # Add optional sum columns if they exist
    if 'xG' in season_df.columns:
        agg_dict['career_xG'] = ('xG', 'sum')
    if 'xA' in season_df.columns:
        agg_dict['career_xA'] = ('xA', 'sum')
    if 'Sh' in season_df.columns:
        agg_dict['career_shots'] = ('Sh', 'sum')
    if 'SoT' in season_df.columns:
        agg_dict['career_shots_on_target'] = ('SoT', 'sum')

    # Optional naive averages for reference (if requested)
    if 'Goal_Conv_Pct' in season_df.columns:
        agg_dict['avg_season_goal_conv_pct'] = ('Goal_Conv_Pct', 'mean')

    # Keep latest values for snapshot traits
    if 'market_value_in_eur' in season_df.columns:
        agg_dict['latest_market_value'] = ('market_value_in_eur', 'last')
    if 'overall' in season_df.columns:
        agg_dict['latest_overall'] = ('overall', 'last')

    # Perform aggregation
    career_agg = season_df.groupby(['canonical_id', 'common_name']).agg(**agg_dict).reset_index()

    # Derived Career Metrics (Calculated from TOTALS, not averaged seasons)
    career_df = career_agg.copy()
    career_df['career_90s'] = career_df['career_minutes'] / 90.0
    mask = career_df['career_90s'] > 0

    # Using standardized names to match season_df
    career_df['Goals_per90'] = np.where(mask, (career_df['career_goals'] / career_df['career_90s']).round(3), np.nan)
    career_df['Assists_per90'] = np.where(mask, (career_df['career_assists'] / career_df['career_90s']).round(3), np.nan)
    career_df['GoalContributions_per90'] = np.where(mask, ((career_df['career_goals'] + career_df['career_assists']) / career_df['career_90s']).round(3), np.nan)
    career_df['mins_per_goal'] = np.where(career_df['career_goals'] > 0, (career_df['career_minutes'] / career_df['career_goals']).round(1), np.nan)

    # Weighted averages from career totals
    if 'career_shots' in career_df.columns and 'career_goals' in career_df.columns:
        career_df['career_goal_conv_pct'] = np.where(career_df['career_shots'] > 0, ((career_df['career_goals'] / career_df['career_shots']) * 100).round(2), np.nan)

    if 'career_shots' in career_df.columns and 'career_xG' in career_df.columns:
        career_df['career_xG_per_shot'] = np.where(career_df['career_shots'] > 0, (career_df['career_xG'] / career_df['career_shots']).round(3), np.nan)

    if 'career_xG' in career_df.columns:
        career_df['career_xG_overperformance'] = (career_df['career_goals'] - career_df['career_xG']).round(3)

    print(f"✅ Built accurate career summary for {len(career_df)} players using weighted totals.")
    display(career_df.head())


📋 CAREER SUMMARY (Summed Totals)
✅ Built accurate career summary for 8 players using weighted totals.


,canonical_id,common_name,career_minutes,career_goals,career_assists,seasons_played,latest_overall,career_90s,Goals_per90,Assists_per90,GoalContributions_per90,mins_per_goal
0,de_bruyne,Kevin De Bruyne,11602,47,68,6,91.0,128.911111,0.365,0.527,0.892,246.9
1,haaland,Erling Haaland,10702,125,29,5,NaN,118.911111,1.051,0.244,1.295,85.6
2,lewandowski,Robert Lewandowski,16717,174,36,6,90.0,185.744444,0.937,0.194,1.131,96.1
3,mbappe,Kylian Mbappé,14235,162,48,6,91.0,158.166667,1.024,0.303,1.328,87.9
4,messi,Lionel Messi,13606,113,73,5,NaN,151.177778,0.747,0.483,1.230,120.4


✅ Cell 8: Sanity Checks (Zero-Minute Bug Fixes)

In [23]:
print("\n" + "=" * 70)
print("🔍 SANITY CHECKS")
print("=" * 70)

# 8.1 Check for duplicate match IDs per player in StatsBomb
if not match_features.empty and 'match_id' in match_features.columns:
    dup_matches = match_features.duplicated(subset=['canonical_id', 'match_id'])
    if dup_matches.any():
        print(f"⚠️ Found {dup_matches.sum()} duplicate match assignments per player.")
        match_features = match_features[~dup_matches].copy()
    else:
        print("✅ No duplicate match assignments per player.")

# 8.2 Check for duplicate seasons per player
if not season_df.empty:
    dup_check = season_df.duplicated(subset=['canonical_id', 'Season'])
    if dup_check.any():
        print(f"⚠️ Found {dup_check.sum()} duplicate player‑season rows.")
        season_df = season_df[~dup_check].copy()
    else:
        print("✅ No duplicate player‑seasons.")

# 8.3 Check negative minutes
if not season_df.empty:
    neg_min = (season_df['Minutes'] < 0).sum()
    if neg_min > 0:
        print(f"⚠️ Found {neg_min} rows with negative minutes.")
        season_df = season_df[season_df['Minutes'] >= 0].copy()
    else:
        print("✅ No negative minutes.")

# 8.4 Check for zero minutes but non‑zero stats
if not season_df.empty:
    bad = season_df[(season_df['Minutes'] == 0) & (season_df['Gls'].fillna(0) > 0)]
    if not bad.empty:
        print(f"⚠️ Found {len(bad)} rows with goals but zero minutes – dropping.")
        season_df = season_df[~((season_df['Minutes'] == 0) & (season_df['Gls'].fillna(0) > 0))].copy()
    else:
        print("✅ No zero‑minute goal scorers.")

# 8.5 Check missing canonical_id
for name, df in [('Events', events), ('FBref', fbref_filtered), ('Season Merge', season_df)]:
    if not df.empty and 'canonical_id' in df.columns:
        missing = df['canonical_id'].isna().sum()
        if missing > 0:
            print(f"⚠️ {name}: {missing} rows missing canonical_id.")
        else:
            print(f"✅ {name}: all rows mapped perfectly to a canonical_id.")


🔍 SANITY CHECKS
✅ No duplicate match assignments per player.
✅ No duplicate player‑seasons.
✅ No negative minutes.
✅ No zero‑minute goal scorers.
✅ Events: all rows mapped perfectly to a canonical_id.
✅ FBref: all rows mapped perfectly to a canonical_id.
✅ Season Merge: all rows mapped perfectly to a canonical_id.


💾 Cell 9: Save Everything

In [25]:
print("\n" + "=" * 70)
print("💾 SAVING PROCESSED DATASETS & METADATA")
print("=" * 70)

import json
import hashlib
from datetime import datetime

# Helper for robust checksums
def get_checksum(df):
    if df.empty: return "N/A"
    return hashlib.md5(pd.util.hash_pandas_object(df, index=True).values).hexdigest()

# 9.1 Match features
if not match_features.empty:
    match_path = OUTPUT / "player_match_stats.parquet"
    match_features.to_parquet(match_path, index=False)
    print(f"✅ Match stats saved to: {match_path}")

# 9.2 Season features
if not season_df.empty:
    season_path = OUTPUT / "player_season_stats.parquet"
    season_df.to_parquet(season_path, index=False)
    print(f"✅ Season stats saved to: {season_path}")

# 9.3 Career summary
if not career_df.empty:
    career_path = OUTPUT / "player_summary.parquet"
    career_df.to_parquet(career_path, index=False)
    print(f"✅ Career summary saved to: {career_path}")

# 9.4 Descriptive Feature Metadata
if not season_df.empty:
    feature_dict = {
        'Goals_per90': 'Goals scored per 90 minutes',
        'xG_per_Shot': 'Expected goals value divided by total shots',
        'Minutes': 'Total minutes played',
        'ShotAccuracy': 'Shots on target divided by total shots',
        'Assists_per90': 'Assists provided per 90 minutes',
        'GoalContributions_per90': 'Goals plus assists per 90 minutes',
        'xg_overperformance': 'Goals minus expected goals',
    }

    meta_data = []
    for col in season_df.columns:
        meta_data.append({
            "feature": col,
            "type": str(season_df[col].dtype),
            "description": feature_dict.get(col, "Pipeline generated statistical feature"),
            "range": f"{season_df[col].min():.2f} to {season_df[col].max():.2f}" if pd.api.types.is_numeric_dtype(season_df[col]) else "categorical"
        })
    meta_df = pd.DataFrame(meta_data)
    meta_path = PATHS["metadata"] / "feature_metadata.csv"
    meta_df.to_csv(meta_path, index=False)

# 9.5 Extended Dataset Inventory
current_time = datetime.utcnow().isoformat()
inventory_df = pd.DataFrame({
    "dataset": ["player_match_stats.parquet", "player_season_stats.parquet", "player_summary.parquet"],
    "rows": [len(match_features) if not match_features.empty else 0,
             len(season_df) if not season_df.empty else 0,
             len(career_df) if not career_df.empty else 0],
    "columns": [len(match_features.columns) if not match_features.empty else 0,
                len(season_df.columns) if not season_df.empty else 0,
                len(career_df.columns) if not career_df.empty else 0],
    "created_at": [current_time] * 3,
    "phase": [2, 2, 2],
    "version": ["1.0", "1.0", "1.0"],
    "checksum": [get_checksum(match_features), get_checksum(season_df), get_checksum(career_df)]
})
inventory_path = PATHS["metadata"] / "dataset_inventory.csv"
inventory_df.to_csv(inventory_path, index=False)
print(f"✅ Extended dataset inventory saved to: {inventory_path}")

# 9.6 Phase Manifest Generation
manifest = {
    "phase": 2,
    "created": current_time,
    "datasets": [
        "player_match_stats.parquet",
        "player_season_stats.parquet",
        "player_summary.parquet"
    ],
    "statsbomb_rows": len(events) if not events.empty else 0,
    "fbref_rows": len(fbref_filtered) if not fbref_filtered.empty else 0,
    "understat_rows": len(understat) if not understat.empty else 0,
    "transfermarkt_rows": len(tm) if not tm.empty else 0,
    "fifa_rows": len(fifa) if not fifa.empty else 0
}
manifest_path = PATHS["metadata"] / "phase2_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"✅ Phase 2 manifest created at: {manifest_path}")


PIPELINE_VERSION = "2.0.0"

# Export explicit execution configuration
pipeline_config = {
    "phase": 2,
    "version": PIPELINE_VERSION,
    "seed": SEED,
    "min_minutes_threshold": 500,
    "created_at": current_time,
    "data_sources": ["statsbomb", "fbref", "understat", "transfermarkt", "fifa"]
}
with open(PATHS["metadata"] / "pipeline_config.json", "w") as f:
    json.dump(pipeline_config, f, indent=2)

print(f"✅ Pipeline config v{PIPELINE_VERSION} saved.")


💾 SAVING PROCESSED DATASETS & METADATA
✅ Match stats saved to: /content/drive/MyDrive/GOAT-Net/data/processed/statistical/player_match_stats.parquet
✅ Season stats saved to: /content/drive/MyDrive/GOAT-Net/data/processed/statistical/player_season_stats.parquet
✅ Career summary saved to: /content/drive/MyDrive/GOAT-Net/data/processed/statistical/player_summary.parquet
✅ Extended dataset inventory saved to: /content/drive/MyDrive/GOAT-Net/metadata/dataset_inventory.csv
✅ Phase 2 manifest created at: /content/drive/MyDrive/GOAT-Net/metadata/phase2_manifest.json
✅ Pipeline config v2.0.0 saved.


📈 Cell 10: Summary Report

In [26]:
# ============================================================
# GOAT-Net — Phase 2: Summary Report (Fixed)
# ============================================================
print("\n" + "=" * 70)
print("📈 FINAL PIPELINE SUMMARY")
print("=" * 70)

if not match_features.empty:
    print(f"Match appearances       : {len(match_features)}")
    print(f"Unique players tracked  : {match_features['canonical_id'].nunique()}")
if not season_df.empty:
    print(f"Player‑seasons          : {len(season_df)}")
    print(f"Seasons range           : {season_df['Season'].min()} to {season_df['Season'].max()}")
if not career_df.empty:
    print(f"Career summaries built  : {len(career_df)}")

    # ---- Compute missing data % ----
    print("\n⚠️ Missing Data Report (% of rows missing source info):")
    us_miss = season_df['understat_xG'].isna().mean() * 100 if 'understat_xG' in season_df.columns else 100
    fifa_miss = season_df['overall'].isna().mean() * 100 if 'overall' in season_df.columns else 100
    tm_miss = season_df['market_value_in_eur'].isna().mean() * 100 if 'market_value_in_eur' in season_df.columns else 100

    print(f"Seasons missing Understat : {us_miss:.1f}%")
    print(f"Seasons missing FIFA      : {fifa_miss:.1f}%")
    print(f"Seasons missing TM Value  : {tm_miss:.1f}%")

    # ---- Prepare columns for display (Using standardized 'Goals_per90') ----
    display_cols = ['common_name', 'career_goals', 'Goals_per90']

    if 'career_xG_overperformance' in career_df.columns:
        display_cols.append('career_xG_overperformance')
    else:
        print("ℹ️ Note: career_xG_overperformance not available – skipping overperformance metric.")

    print("\nTop 3 Players by Career Goals per 90:")

    # Ensure display_cols only includes columns that actually exist in career_df
    valid_display_cols = [c for c in display_cols if c in career_df.columns]

    top_goal_players = career_df[valid_display_cols].sort_values('Goals_per90', ascending=False).head(3)
    display(top_goal_players)

print("\n🎉 PHASE 2 STATISTICAL ENGINE COMPLETE! Data is ready for machine learning.")


📈 FINAL PIPELINE SUMMARY
Match appearances       : 885
Unique players tracked  : 7
Player‑seasons          : 44
Seasons range           : 2018-2019 to 2023-2024
Career summaries built  : 8

⚠️ Missing Data Report (% of rows missing source info):
Seasons missing Understat : 27.3%
Seasons missing FIFA      : 22.7%
Seasons missing TM Value  : 100.0%
ℹ️ Note: career_xG_overperformance not available – skipping overperformance metric.

Top 3 Players by Career Goals per 90:


,common_name,career_goals,Goals_per90
1,Erling Haaland,125,1.051
3,Kylian Mbappé,162,1.024
2,Robert Lewandowski,174,0.937



🎉 PHASE 2 STATISTICAL ENGINE COMPLETE! Data is ready for machine learning.


##cell 11

In [27]:
# ============================================================
# Cell 11: Phase 2 GitHub Sync (Direct from Drive)
# ============================================================
import os
import subprocess
from pathlib import Path
from google.colab import userdata
from IPython.display import display, Javascript

# Force Colab to automatically save the notebook right before pushing
display(Javascript('IPython.notebook.save_checkpoint();'))

# We are already in /content/drive/MyDrive/GOAT-Net thanks to Cell 1
REPO = Path("/content/drive/MyDrive/GOAT-Net")

def run_git(args, **kwargs):
    return subprocess.run(["git"] + args, cwd=REPO, capture_output=True, text=True, **kwargs)

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    if not GITHUB_TOKEN:
        raise ValueError("empty secret")

    REPO_SLUG = "AtiX-Algo/GOAT-Net"
    CLEAN_URL = f"https://github.com/{REPO_SLUG}.git"
    AUTH_URL = f"https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git"

    # --- SELF-HEALING CHECK ---
    # If Drive lost the .git folder, re-initialize it instantly
    if not (REPO / ".git").exists():
        print("⚠️ .git folder missing from Drive. Re-initializing repository...")
        run_git(["init"])
        run_git(["remote", "add", "origin", CLEAN_URL])
        run_git(["branch", "-M", "main"])

    # Configure Git
    run_git(["config", "--global", "user.email", "atix.algo@gmail.com"])
    run_git(["config", "--global", "user.name", "AtiX-Algo"])

    # Stage code, docs, and lightweight metadata directly from Drive
    print("📦 Staging files for commit...")
    run_git(["add", "notebooks/"])
    run_git(["add", "src/"])
    run_git(["add", "metadata/"])
    run_git(["add", "docs/"])
    run_git(["add", "requirements.txt"])

    # Commit
    commit_res = run_git(["commit", "-m", "feat: finalize Phase 2 architecture & generate manifests"])
    if "nothing to commit" in commit_res.stdout:
        print("ℹ️ No new changes to commit. Proceeding to push anyway...")
    else:
        print(commit_res.stdout)

    # FIX GIT STATE & SYNC
    run_git(["remote", "set-url", "origin", AUTH_URL], check=True)

    # Abort any broken rebases
    subprocess.run(["rm", "-rf", ".git/rebase-merge"], cwd=REPO, capture_output=True)
    run_git(["rebase", "--abort"])

    # Force the local 'main' branch to attach to HEAD
    run_git(["branch", "-f", "main", "HEAD"])
    run_git(["checkout", "main"])

    print("\n🔄 Syncing with GitHub (Pulling remote changes)...")
    env = os.environ.copy()
    env["GIT_MERGE_AUTOEDIT"] = "no"

    pull_cmd = ["git", "pull", "origin", "main", "--no-rebase", "-s", "recursive", "-X", "ours", "--allow-unrelated-histories"]
    pull_result = subprocess.run(pull_cmd, cwd=REPO, capture_output=True, text=True, env=env)

    print("🚀 Pushing GOAT-Net updates to GitHub...")
    push_result = run_git(["push", "-u", "origin", "main", "--force"])

    # Strip the token back out immediately for security
    run_git(["remote", "set-url", "origin", CLEAN_URL], check=True)

    if push_result.returncode == 0:
        print("\n✅ Pushed successfully to GitHub. Your repository is now perfectly in sync!")
    else:
        print("\n❌ Push failed. Exact error from GitHub:")
        safe_error = push_result.stderr.replace(GITHUB_TOKEN, "***HIDDEN_TOKEN***")
        print(safe_error)

except Exception as e:
    token_str = GITHUB_TOKEN if 'GITHUB_TOKEN' in locals() and GITHUB_TOKEN else "UNKNOWN_TOKEN"
    safe_error = str(e).replace(token_str, "***HIDDEN_TOKEN***")
    print(f"\n⚠️ GitHub Sync Skipped: {safe_error}")

<IPython.core.display.Javascript object>

📦 Staging files for commit...
[main e5fd9e9] feat: finalize Phase 2 architecture & generate manifests
 3 files changed, 6 insertions(+), 45 deletions(-)
 rewrite metadata/dataset_inventory.csv (100%)


🔄 Syncing with GitHub (Pulling remote changes)...
🚀 Pushing GOAT-Net updates to GitHub...

✅ Pushed successfully to GitHub. Your repository is now perfectly in sync!


In [ ]:
# !cd /content/GOAT-Net && \
# echo "===== GIT STATUS =====" && \
# git status && \
# echo && \
# echo "===== SRC FILES =====" && \
# git ls-files | grep "^src/" || true && \
# echo && \
# echo "===== NOTEBOOK FILES =====" && \
# git ls-files | grep "^notebooks/" || true && \
# echo && \
# echo "===== GITIGNORE =====" && \
# cat .gitignore

In [ ]:
!cd /content/GOAT-Net && ls -la

In [ ]:
!cd /content/GOAT-Net && git add src notebooks
!cd /content/GOAT-Net && git status

In [ ]:
# !cd /content/GOAT-Net && \
# git restore --staged src/data/__pycache__/dataset_manager.cpython-312.pyc && \
# rm -rf src/data/__pycache__

In [ ]:
# !cd /content/GOAT-Net && git add .gitignore

In [ ]:
# !cd /content/GOAT-Net && \
# git commit -m "feat: add project source code and Phase 2-3 notebooks"

# !cd /content/GOAT-Net && \
# git push origin main